# 02. Loading Engineering Datasets
**Engineering Data Processing with Python** | one-day course | about 35 minutes

> **How this notebook works**
> * Run every grey code cell in order: click it, then press **Shift + Enter**.
> * Most cells just need running and reading. Cells marked **Your turn** have a blank, `____`, for you to replace. The example just above shows you how.
> * Stuck? Click **Stuck? Click to show the answer** under the cell. Using it is fine.
> * Cells that show only a title are helper code written for you. Just run them.


In [ ]:
# @title Setup: run this cell first (it loads the course data)
# ===================== SETUP: run this cell first (click it, then Shift+Enter) =====================
# It makes sure the course data is available in this Colab session and loads the tools we need.
import os, zipfile, urllib.request
from pathlib import Path

DATA_ZIP_URL = "https://raw.githubusercontent.com/buildyourai/ei-python-data-processing/main/data.zip"   # course data (fictional / public)

def get_course_data():
    if Path("data").is_dir():
        print("Data folder found:", sorted(p.name for p in Path("data").iterdir())[:20])
        return
    if not Path("data.zip").exists() and DATA_ZIP_URL:
        print("Downloading course data...")
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    zip_path = Path("data.zip")
    if not zip_path.exists():
        try:
            from google.colab import files
            print("Choose data.zip from your computer (your trainer will tell you where it is).")
            uploaded = files.upload()
            zip_path = Path(next(iter(uploaded)))
        except ImportError:
            raise FileNotFoundError("Put data.zip or the data folder next to this notebook.")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    print("Data ready:", sorted(p.name for p in Path("data").iterdir()))

get_course_data()
Path("outputs").mkdir(exist_ok=True)

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

def check(condition, ok="Looks right.", hint="Not quite yet. Re-read the instructions above the cell."):
    """Tiny self-check helper used throughout the course."""
    print(("PASS: " + ok) if condition else ("NOT YET: " + hint))

## The point of this module

pandas will load almost anything without an error. **Loading without an error does not mean loading correctly.**
You will load a CSV and two Excel files, get the dates right, and find a fault that no error message will report.

## 1. A year of energy readings from a steel plant (35,000 rows)

In [ ]:
steel = pd.read_csv("data/steel.csv")
print(steel.shape)
steel.head()

In [ ]:
steel.dtypes

`date` is `object` (text). To sort by time or total by day, it must become a real date.

### The day/month trap
Irish exports write day/month/year. Left to guess, pandas often assumes month/day. Watch the 1st of February:

In [ ]:
print(pd.to_datetime("01/02/2018 00:15"))                            # guessed: 2 January. Wrong, no error.
print(pd.to_datetime("01/02/2018 00:15", format="%d/%m/%Y %H:%M"))   # told: 1 February. Right.

Always tell pandas the format. The codes: `%d` day, `%m` month, `%Y` year, `%H` hour, `%M` minute.

### Your turn 2.1
Convert the steel dates. The format is `%d/%m/%Y %H:%M`, exactly as in the example above.

In [ ]:
steel["date"] = pd.to_datetime(steel["date"], format="____")
print("From", steel["date"].min(), "to", steel["date"].max())

<details><summary><b>Stuck? Click to show the answer</b></summary>

```python
steel["date"] = pd.to_datetime(steel["date"], format="%d/%m/%Y %H:%M")
print("From", steel["date"].min(), "to", steel["date"].max())
```
</details>

In [ ]:
check(steel["date"].dt.month.nunique() == 12, "All 12 months present.", "Check the format: day first, then month.")

## 2. A fault nobody will tell you about

Readings should move forwards in time. Do they? Run and read.

In [ ]:
print("Does time always go forwards?", steel["date"].is_monotonic_increasing)
steel.loc[94:97, ["date", "Usage_kWh", "NSM"]]

After 23:45 on 1 January comes **00:00 on 1 January**, then 00:15 on 2 January. The midnight reading has the wrong
date: it belongs to the next day. This happens every day, 365 times, with no error. (`NSM`, seconds since midnight,
is 0 on that row, which confirms it.)

The fix: add one day to every reading stamped exactly 00:00. Run and read.

In [ ]:
midnight = (steel["date"].dt.hour == 0) & (steel["date"].dt.minute == 0)
steel.loc[midnight, "date"] = steel.loc[midnight, "date"] + pd.Timedelta(days=1)
print("Does time always go forwards now?", steel["date"].is_monotonic_increasing)

Lesson: check that time series go forwards and have even steps. Nobody else will.

## 3. Daily and weekly totals

`resample("D")` adds up readings by day (like a pivot table on dates).

In [ ]:
daily = steel.set_index("date")["Usage_kWh"].resample("D").sum()
daily.plot(title="Daily energy use (kWh)", figsize=(10, 3));

### Your turn 2.2
Change `"D"` (day) to `"W"` (week) to plot weekly totals.

In [ ]:
weekly = steel.set_index("date")["Usage_kWh"].resample("____").sum()
weekly.plot(title="Weekly energy use (kWh)", figsize=(10, 3));

<details><summary><b>Stuck? Click to show the answer</b></summary>

```python
weekly = steel.set_index("date")["Usage_kWh"].resample("W").sum()
weekly.plot(title="Weekly energy use (kWh)", figsize=(10, 3));
```
</details>

## 4. Excel with title rows: the Kilbrack asset register

Load it the simple way first and look at the column names.

In [ ]:
pd.read_excel("data/asset_register.xlsx").head(6)

The real header row (`Asset ID`, `Site`...) is the 4th row. Above it are a title, a note and a blank row.
`skiprows` tells pandas how many rows to skip.

### Your turn 2.3
How many rows should be skipped? Count the junk rows above the header in the output above.

In [ ]:
register = pd.read_excel("data/asset_register.xlsx", skiprows=____)
register.head()

<details><summary><b>Stuck? Click to show the answer</b></summary>

```python
register = pd.read_excel("data/asset_register.xlsx", skiprows=3)
register.head()
```
</details>

In [ ]:
check("Asset ID" in register.columns, "Header found.", "Expected a column called 'Asset ID'. Try a different number.")

Excel files can have several sheets. This one has a second sheet called `Decommissioned`.

### Your turn 2.4
Load the second sheet by name.

In [ ]:
decommissioned = pd.read_excel("data/asset_register.xlsx", sheet_name="____")
decommissioned

<details><summary><b>Stuck? Click to show the answer</b></summary>

```python
decommissioned = pd.read_excel("data/asset_register.xlsx", sheet_name="Decommissioned")
decommissioned
```
</details>

## 5. The work orders, loaded properly

Two settings matter here:
* `na_values=["-"]`: this export uses `-` to mean "nothing". Blanks and `N/A` are treated as missing automatically.
* Dates: most are `dd/mm/yyyy HH:MM`, but a few were typed as `yyyy-mm-dd HH:MM`.

The helper below tries a list of formats in order, and tells you how many dates did not fit any of them.
Just run it.

In [ ]:
# @title Helper: parse_dates(column, list_of_formats) (just run it)
def parse_dates(series, formats):
    result = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")
    for fmt in formats:
        result = result.fillna(pd.to_datetime(series, format=fmt, errors="coerce"))
    print(f"{result.isna().sum()} dates did not match any format")
    return result
print("parse_dates() is ready")

In [ ]:
wo = pd.read_csv("data/work_orders_cmms.csv", na_values=["-"])

# Try with only the main format first
raised = parse_dates(wo["date_raised"], ["%d/%m/%Y %H:%M"])
wo.loc[raised.isna(), "date_raised"].head()

### Your turn 2.5
Add the second format to the list: `%Y-%m-%d %H:%M` (year first, with dashes).

In [ ]:
wo["date_raised"] = parse_dates(wo["date_raised"], ["%d/%m/%Y %H:%M", "____"])

<details><summary><b>Stuck? Click to show the answer</b></summary>

```python
wo["date_raised"] = parse_dates(wo["date_raised"], ["%d/%m/%Y %H:%M", "%Y-%m-%d %H:%M"])
```
</details>

In [ ]:
check(wo["date_raised"].isna().sum() == 0, "Every date parsed, and you know exactly which formats were in the file.",
      "Some dates still did not parse. Check the second format.")

Why not let pandas guess? Guessing is fine for a quick look. For something that runs every month you want it to
**tell you** when a new format appears, not make a silent guess.

## Optional extras

**Thermocouple data with a two-row header.** Run and look: the cooling curves of an aluminium alloy, from a sheet
with a title, a row of sensor positions, then the real header.

In [ ]:
thermal = pd.read_excel("data/thermal-profiles.xlsx", sheet_name="Al-11wt.%Si", header=3)
thermal.set_index(thermal.columns[0]).plot(figsize=(10, 4), title="Cooling curves, Al-11wt.%Si", ylabel="°C");

**With Gemini:** `data/manifest.TXT` and `data/manifest.csv` both list the same microscopy files. Ask Gemini to help
you load the TXT into a table and compare the two. Do they agree? (Hint: count the files, and look at `TEMgrid06.TIF`.)

## Recap
* Tell `pd.to_datetime` the format. Day/month mix-ups give no error.
* Check time series go forwards.
* `skiprows` and `sheet_name` handle most Excel layouts.
* `na_values` adds extra "missing" markers.

Next: **03 Cleaning and Standardising**.